# R9 — Monte Carlo Dropout Confidence / Abstention Test

Confidence/abstention ablation only: the underlying NN v1/NN v2 BCE model (R1 cold-restart cadence, 34 engineered features, `CARRY_FORWARD_BCE`) is never retrained -- MC dropout is used only to decide whether to keep or reject a deterministic Top-2/Top-3 candidate. Ranking is always by the deterministic score; MC statistics never rerank and rejected slots are never backfilled from lower-ranked sectors. Built on `TOP2`/`TOP3` (R1's/R8's validated simple selection), explicitly NOT on R8's dynamic ROC thresholds (`R8_NEGATIVE`). See `docs/weekly_sector_rotation_r9_mc_dropout_test.md` for the full write-up and `r9_mc_dropout.py` for the shared, unit-tested stochastic-inference/confidence implementations this notebook imports. Does not implement R10.

In [1]:
"""
R9 full pipeline (scratchpad validation run). Once verified, transcribed into
r9_mc_dropout_abstention.ipynb and executed there for the committed
artifacts. Confidence/abstention ablation only: the underlying NN v1/v2 BCE
model (R1 cold-restart cadence, CARRY_FORWARD_BCE) is never retrained --
MC dropout is used only to decide whether to keep or reject a deterministic
Top-2/Top-3 candidate. Ranking is always by deterministic score.
"""
import json
import sys
import time
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

t0 = time.time()
REPO_ROOT = Path("/Users/gavinhussey/Downloads/AtlasQuant")
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"

sys.path.insert(0, str(REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation"))
import r7_financial_losses as r7  # noqa: E402
import r9_mc_dropout as r9  # noqa: E402

SECTOR_ETFS = r7.SECTOR_ETFS
FULL_FEATURE_COLS = r7.FULL_FEATURE_COLS
BLOCK_SIZE = r7.BLOCK_SIZE
N_BOOTSTRAP = r7.N_BOOTSTRAP
BOOTSTRAP_SEED = r7.BOOTSTRAP_SEED
K_VALUES = [2, 3]

# ---------------------------------------------------------------------------

## 1. Load panel + R1 primary target, authoritative feature set, and reproduce/verify R1's exact fold schedule.

In [2]:
# 1. Load panel + R1 target, authoritative feature set, fold schedule.
# ---------------------------------------------------------------------------
date_cols = ["date", "week_end", "actual_last_trading_date", "feature_cutoff_timestamp",
             "next_week_first_trading_date", "next_week_last_trading_date"]
panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=date_cols)
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)
AUDIT_COLUMNS = ["week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
                  "next_week_first_trading_date", "next_week_last_trading_date"]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

r1_targets = pd.read_csv(OUTPUTS_ROOT / "r1_target_definitions_full_panel.csv", parse_dates=["date"])
panel = panel.merge(
    r1_targets[["date", "symbol", "target_abs_1pct_next_week", "next_week_open_to_close_return",
                "next_week_spy_return"]],
    on=["date", "symbol"], how="left",
)
assert set(FULL_FEATURE_COLS).issubset(set(FEATURE_COLUMNS))

md = panel.dropna(subset=FEATURE_COLUMNS + ["target_abs_1pct_next_week"]).copy()
md["label_binary"] = md["target_abs_1pct_next_week"].astype(int)
md["year"] = md["date"].dt.year
years_all = sorted(md["year"].unique())
assert years_all == list(range(2005, 2027))
test_years = r7.derive_r1_fold_schedule(years_all)
assert test_years == list(range(2008, 2027)), f"fold schedule diverges from validated R1 schedule: {test_years}"
print(f"[{time.time()-t0:.0f}s] data loaded, {len(md)} rows, test_years={test_years[0]}-{test_years[-1]}")

# ---------------------------------------------------------------------------

[0s] data loaded, 12253 rows, test_years=2008-2026


## 2. Model definitions -- verbatim R1/R7/R8 `SmallMLP` (NN v1) / `WideSingleLayerMLP` seed-ensemble (NN v2).

In [3]:
# 2. Model definitions -- verbatim R1/R7/R8.
# ---------------------------------------------------------------------------
NN_SEED = 20260812
HIDDEN_1, HIDDEN_2 = 16, 8
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 300
PATIENCE = 15


class SmallMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


V2_HIDDEN = 32
V2_DROPOUT = 0.4
V2_WEIGHT_DECAY = 1e-3
V2_LEARNING_RATE = 1e-3
V2_MAX_EPOCHS = 300
V2_PATIENCE = 15
N_SEEDS = 5
SEED_BASE = 20260812


class WideSingleLayerMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN), nn.ReLU(), nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


BCE_LOSS = nn.BCEWithLogitsLoss()


def train_one_model(X_fit, y_fit, X_val, y_val, n_features, seed, v2=False):
    torch.manual_seed(seed)
    model = WideSingleLayerMLP(n_features) if v2 else SmallMLP(n_features)
    lr = V2_LEARNING_RATE if v2 else LEARNING_RATE
    wd = V2_WEIGHT_DECAY if v2 else WEIGHT_DECAY
    max_epochs = V2_MAX_EPOCHS if v2 else MAX_EPOCHS
    patience = V2_PATIENCE if v2 else PATIENCE
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32); y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(max_epochs):
        model.train(); optimizer.zero_grad()
        loss = BCE_LOSS(model(X_fit_t), y_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = BCE_LOSS(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    return model


print(f"[{time.time()-t0:.0f}s] Step 2 (model defs) done.")

# ---------------------------------------------------------------------------

[0s] Step 2 (model defs) done.


## 3. Walk-forward training (verbatim R1/R7/R8 cold-restart cadence) + 100-pass MC-dropout inference on the SAME trained model(s), per fold, before discarding them -- then verify deterministic pooled AUC reproduces R1/R7/R8's saved values exactly.

In [4]:
# 3. Walk-forward: train (verbatim R1/R7/R8), get deterministic OOF, then run
#    100-pass (+ 50-pass subset) MC dropout inference on the SAME trained
#    model(s) before discarding them for the next fold.
# ---------------------------------------------------------------------------
def run_walk_forward_with_mc(md, years_, test_years_, feature_cols, model_name, v2=False):
    oof_frames = []
    mc_summary_frames = []
    n_features = len(feature_cols)
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        train_mask = md["year"].isin(train_years_available)
        fit_mask = md["year"].isin(fit_years)
        val_mask = md["year"] == val_year
        test_mask = md["year"] == test_year
        scaler = StandardScaler().fit(md.loc[train_mask, feature_cols])
        X_fit = scaler.transform(md.loc[fit_mask, feature_cols])
        X_val = scaler.transform(md.loc[val_mask, feature_cols])
        X_test = scaler.transform(md.loc[test_mask, feature_cols])
        y_fit = md.loc[fit_mask, "label_binary"].to_numpy()
        y_val = md.loc[val_mask, "label_binary"].to_numpy()
        X_test_t = torch.tensor(X_test, dtype=torch.float32)
        n_test = X_test.shape[0]

        if v2:
            models = [train_one_model(X_fit, y_fit, X_val, y_val, n_features, SEED_BASE + m, v2=True)
                      for m in range(N_SEEDS)]
            with torch.no_grad():
                det_preds = np.mean([torch.sigmoid(m(X_test_t)).numpy() for m in models], axis=0)
            member_passes = np.empty((N_SEEDS, r9.MC_PASSES_PRIMARY, n_test))
            for member_idx, model in enumerate(models):
                r9.enable_mc_dropout(model)
                for pass_idx in range(r9.MC_PASSES_PRIMARY):
                    torch.manual_seed(r9.mc_pass_seed(model_name, test_year, member_idx, pass_idx))
                    with torch.no_grad():
                        member_passes[member_idx, pass_idx, :] = torch.sigmoid(model(X_test_t)).numpy()
            all_passes = member_passes.mean(axis=0)  # (MC_PASSES_PRIMARY, n_test) -- ensemble mean per pass
        else:
            model = train_one_model(X_fit, y_fit, X_val, y_val, n_features, NN_SEED, v2=False)
            with torch.no_grad():
                det_preds = torch.sigmoid(model(X_test_t)).numpy()
            r9.enable_mc_dropout(model)
            all_passes = np.empty((r9.MC_PASSES_PRIMARY, n_test))
            for pass_idx in range(r9.MC_PASSES_PRIMARY):
                torch.manual_seed(r9.mc_pass_seed(model_name, test_year, 0, pass_idx))
                with torch.no_grad():
                    all_passes[pass_idx, :] = torch.sigmoid(model(X_test_t)).numpy()

        fold = md.loc[test_mask, ["date", "symbol", "next_week_open_to_close_return",
                                   "next_week_spy_return", "label_binary"]].copy()
        fold["predicted_proba"] = det_preds
        oof_frames.append(fold)

        rows = []
        for i in range(n_test):
            p_det = det_preds[i]
            passes_100 = all_passes[:, i]
            passes_50 = passes_100[:r9.MC_PASSES_STABILITY]  # first 50 of the same 100 (Step 22)
            s100 = r9.mc_summary_stats(passes_100, p_det)
            cm100 = r9.confidence_mass(passes_100, median=s100["mc_median"], std=s100["mc_std"])
            s50 = r9.mc_summary_stats(passes_50, p_det)
            cm50 = r9.confidence_mass(passes_50, median=s50["mc_median"], std=s50["mc_std"])
            row = {"date": fold["date"].iloc[i], "symbol": fold["symbol"].iloc[i], "p_det": p_det,
                   **s100, "confidence_mass": cm100, "mc_confident": r9.is_mc_confident(cm100),
                   "mc_std_50": s50["mc_std"], "mc_median_50": s50["mc_median"],
                   "confidence_mass_50": cm50, "mc_confident_50": r9.is_mc_confident(cm50)}
            rows.append(row)
        mc_summary_frames.append(pd.DataFrame(rows))
        print(f"[{time.time()-t0:.0f}s] {model_name} test_year={test_year}: MC inference done "
              f"({r9.MC_PASSES_PRIMARY} passes x {n_test} rows{' x 5 members' if v2 else ''})")

    oof = pd.concat(oof_frames, ignore_index=True)
    mc_summary = pd.concat(mc_summary_frames, ignore_index=True)
    return oof, mc_summary


oof_v1, mc_v1 = run_walk_forward_with_mc(md, years_all, test_years, FULL_FEATURE_COLS, "NN_v1", v2=False)
auc_v1 = roc_auc_score(oof_v1.label_binary, oof_v1.predicted_proba)
print(f"[{time.time()-t0:.0f}s] NN_v1 deterministic pooled_auc={auc_v1:.6f} (expected 0.568010)")
assert abs(auc_v1 - 0.5680100994943317) < 1e-4, "NN v1 deterministic AUC does not reproduce R1/R7/R8"

oof_v2, mc_v2 = run_walk_forward_with_mc(md, years_all, test_years, FULL_FEATURE_COLS, "NN_v2", v2=True)
auc_v2 = roc_auc_score(oof_v2.label_binary, oof_v2.predicted_proba)
print(f"[{time.time()-t0:.0f}s] NN_v2 deterministic pooled_auc={auc_v2:.6f} (expected 0.573573)")
assert abs(auc_v2 - 0.5735730758971164) < 1e-4, "NN v2 deterministic AUC does not reproduce R1/R7/R8"
print(f"[{time.time()-t0:.0f}s] Step 3 (walk-forward + MC inference, both models) done.")

# ---------------------------------------------------------------------------

[1s] NN_v1 test_year=2008: MC inference done (100 passes x 572 rows)
[1s] NN_v1 test_year=2009: MC inference done (100 passes x 572 rows)


[1s] NN_v1 test_year=2010: MC inference done (100 passes x 583 rows)


[1s] NN_v1 test_year=2011: MC inference done (100 passes x 572 rows)


[2s] NN_v1 test_year=2012: MC inference done (100 passes x 572 rows)


[2s] NN_v1 test_year=2013: MC inference done (100 passes x 572 rows)


...

[22s] NN_v2 test_year=2022: MC inference done (100 passes x 572 rows x 5 members)


[24s] NN_v2 test_year=2023: MC inference done (100 passes x 572 rows x 5 members)


[26s] NN_v2 test_year=2024: MC inference done (100 passes x 572 rows x 5 members)


[27s] NN_v2 test_year=2025: MC inference done (100 passes x 572 rows x 5 members)


[28s] NN_v2 test_year=2026: MC inference done (100 passes x 340 rows x 5 members)
[28s] NN_v2 deterministic pooled_auc=0.573573 (expected 0.573573)
[28s] Step 3 (walk-forward + MC inference, both models) done.


## 4. Merge deterministic scores with MC summary statistics into the full-universe (every ETF, every week) prediction+uncertainty table.

In [5]:
# 4. Merge deterministic OOF with MC summary stats -- the full-universe
#    (every ETF, every week) prediction+uncertainty table per model.
# ---------------------------------------------------------------------------
def merge_full(oof, mc):
    full = oof.merge(mc, on=["date", "symbol"], how="inner")
    assert len(full) == len(oof) == len(mc)
    assert np.allclose(full["predicted_proba"], full["p_det"])
    return full


full_v1 = merge_full(oof_v1, mc_v1)
full_v2 = merge_full(oof_v2, mc_v2)
FULL_BY_MODEL = {"NN_v1": full_v1, "NN_v2": full_v2}

MC_SUMMARY_COLS = ["mc_mean", "mc_median", "mc_std", "mc_p05", "mc_p25", "mc_p75", "mc_p95",
                   "mc_iqr", "mean_minus_deterministic", "confidence_mass", "mc_confident",
                   "mc_std_50", "mc_median_50", "confidence_mass_50", "mc_confident_50"]

mc_prediction_summary_rows = []
for model_name, full in FULL_BY_MODEL.items():
    tmp = full[["date", "symbol", "p_det", "label_binary"] + MC_SUMMARY_COLS].copy()
    tmp.insert(0, "model", model_name)
    mc_prediction_summary_rows.append(tmp)
r9_mc_prediction_summary = pd.concat(mc_prediction_summary_rows, ignore_index=True)
print(f"[{time.time()-t0:.0f}s] Step 4 (full-universe MC summary table) done. "
      f"Overall confidence pass rate: NN_v1={full_v1.mc_confident.mean():.4f} "
      f"NN_v2={full_v2.mc_confident.mean():.4f}")

# ---------------------------------------------------------------------------

[28s] Step 4 (full-universe MC summary table) done. Overall confidence pass rate: NN_v1=0.0008 NN_v2=0.0000


## 5. Build deterministic Top-2/Top-3 candidate tables and the per-candidate MC confidence decisions.

In [6]:
# 5. Build deterministic Top-K candidate tables (K=2,3), both models.
# ---------------------------------------------------------------------------
def build_candidates(full: pd.DataFrame, k: int) -> pd.DataFrame:
    rows = []
    for date, g in full.groupby("date"):
        if len(g) < 11:
            continue
        ranked = g["predicted_proba"].rank(ascending=False, method="first")
        topk = g.loc[ranked <= k].copy()
        topk["rank"] = ranked.loc[ranked <= k].astype(int).to_numpy()
        rows.append(topk)
    return pd.concat(rows, ignore_index=True)


candidates = {}
for model_name, full in FULL_BY_MODEL.items():
    for k in K_VALUES:
        candidates[(model_name, k)] = build_candidates(full, k)
print(f"[{time.time()-t0:.0f}s] Step 5 (Top-K candidate tables) done.")

confidence_decision_rows = []
for (model_name, k), cand in candidates.items():
    tmp = cand[["date", "symbol", "rank", "p_det", "label_binary", "confidence_mass", "mc_confident"]].copy()
    tmp.insert(0, "K", k); tmp.insert(0, "model", model_name)
    tmp["accepted"] = tmp["mc_confident"]
    confidence_decision_rows.append(tmp)
r9_confidence_decisions = pd.concat(confidence_decision_rows, ignore_index=True)

# ---------------------------------------------------------------------------

[30s] Step 5 (Top-K candidate tables) done.


## 6/7. Coverage/abstention metrics and confidence pass-rate breakdowns (by ETF, by year, distribution of confidence_mass).

In [7]:
# 6. Coverage / abstention metrics (Step 13) + confidence pass-rate breakdowns
#    (Step 7: by ETF, by year, by model, distribution of confidence_mass).
# ---------------------------------------------------------------------------
coverage_rows = []
pass_rate_by_etf_rows = []
pass_rate_by_year_rows = []
confidence_mass_dist_rows = []
for (model_name, k), cand in candidates.items():
    n_candidates = len(cand)
    n_accepted = int(cand["mc_confident"].sum())
    per_week_accepted = cand.groupby("date")["mc_confident"].sum()
    n_weeks = len(per_week_accepted)
    coverage_rows.append({
        "model": model_name, "K": k, "n_candidate_etf_weeks": n_candidates,
        "n_accepted_etf_weeks": n_accepted, "n_rejected_etf_weeks": n_candidates - n_accepted,
        "acceptance_rate": n_accepted / n_candidates if n_candidates else np.nan,
        "rejection_rate": (n_candidates - n_accepted) / n_candidates if n_candidates else np.nan,
        "mean_accepted_per_week": per_week_accepted.mean(), "median_accepted_per_week": per_week_accepted.median(),
        "pct_weeks_0_accepted": (per_week_accepted == 0).mean(),
        "pct_weeks_1_accepted": (per_week_accepted == 1).mean(),
        "pct_weeks_fullK_accepted": (per_week_accepted == k).mean(),
        "active_week_pct": (per_week_accepted >= 1).mean(), "n_weeks": n_weeks,
    })

for model_name, full in FULL_BY_MODEL.items():
    by_etf = full.groupby("symbol")["mc_confident"].agg(["mean", "count"]).reset_index()
    by_etf.columns = ["symbol", "pass_rate", "n_obs"]
    by_etf.insert(0, "model", model_name)
    pass_rate_by_etf_rows.append(by_etf)

    by_year = full.copy(); by_year["year"] = by_year["date"].dt.year
    yr = by_year.groupby("year")["mc_confident"].agg(["mean", "count"]).reset_index()
    yr.columns = ["year", "pass_rate", "n_obs"]
    yr.insert(0, "model", model_name)
    pass_rate_by_year_rows.append(yr)

    dist = full["confidence_mass"].describe(percentiles=[0.05, 0.25, 0.5, 0.75, 0.95])
    confidence_mass_dist_rows.append({"model": model_name, **dist.to_dict()})

r9_coverage = pd.DataFrame(coverage_rows)
r9_pass_rate_by_etf = pd.concat(pass_rate_by_etf_rows, ignore_index=True)
r9_pass_rate_by_year = pd.concat(pass_rate_by_year_rows, ignore_index=True)
r9_confidence_mass_distribution = pd.DataFrame(confidence_mass_dist_rows)
print(f"[{time.time()-t0:.0f}s] Step 6/7 (coverage + pass-rate breakdowns) done.")
print(r9_coverage[["model", "K", "acceptance_rate", "mean_accepted_per_week", "active_week_pct"]].to_string())

# ---------------------------------------------------------------------------

[30s] Step 6/7 (coverage + pass-rate breakdowns) done.
   model  K  acceptance_rate  mean_accepted_per_week  active_week_pct
0  NN_v1  2         0.000516                0.001032         0.001032
1  NN_v1  3         0.000688                0.002064         0.002064
2  NN_v2  2         0.000000                0.000000         0.000000
3  NN_v2  3         0.000000                0.000000         0.000000


## 7. Accepted vs. rejected: classification quality and realized-return diagnostics.

In [8]:
# 7. Accepted vs rejected: classification quality (Step 14) + realized
#    returns (Step 15).
# ---------------------------------------------------------------------------
accepted_vs_rejected_rows = []
for (model_name, k), cand in candidates.items():
    base_rate = FULL_BY_MODEL[model_name]["label_binary"].mean()
    accepted = cand[cand["mc_confident"]]
    rejected = cand[~cand["mc_confident"]]

    def _ret_stats(df, prefix):
        if len(df) == 0:
            return {f"{prefix}_n": 0, f"{prefix}_positive_rate": np.nan, f"{prefix}_precision_minus_base": np.nan,
                    f"{prefix}_mean_return": np.nan, f"{prefix}_median_return": np.nan,
                    f"{prefix}_mean_minus_spy": np.nan, f"{prefix}_win_rate_vs_spy": np.nan,
                    f"{prefix}_mean_score": np.nan, f"{prefix}_mean_mc_std": np.nan,
                    f"{prefix}_mean_confidence_mass": np.nan}
        excess = df["next_week_open_to_close_return"] - df["next_week_spy_return"]
        return {
            f"{prefix}_n": len(df), f"{prefix}_positive_rate": float(df["label_binary"].mean()),
            f"{prefix}_precision_minus_base": float(df["label_binary"].mean() - base_rate),
            f"{prefix}_mean_return": float(df["next_week_open_to_close_return"].mean()),
            f"{prefix}_median_return": float(df["next_week_open_to_close_return"].median()),
            f"{prefix}_mean_minus_spy": float(excess.mean()),
            f"{prefix}_win_rate_vs_spy": float((df["next_week_open_to_close_return"] > df["next_week_spy_return"]).mean()),
            f"{prefix}_mean_score": float(df["p_det"].mean()), f"{prefix}_mean_mc_std": float(df["mc_std"].mean()),
            f"{prefix}_mean_confidence_mass": float(df["confidence_mass"].mean()),
        }

    row = {"model": model_name, "K": k}
    row.update(_ret_stats(accepted, "accepted"))
    row.update(_ret_stats(rejected, "rejected"))
    row["accepted_minus_rejected_return"] = row["accepted_mean_return"] - row["rejected_mean_return"] \
        if not (np.isnan(row["accepted_mean_return"]) or np.isnan(row["rejected_mean_return"])) else np.nan
    row["accepted_minus_rejected_excess_spy"] = row["accepted_mean_minus_spy"] - row["rejected_mean_minus_spy"] \
        if not (np.isnan(row["accepted_mean_minus_spy"]) or np.isnan(row["rejected_mean_minus_spy"])) else np.nan
    accepted_vs_rejected_rows.append(row)

r9_accepted_vs_rejected = pd.DataFrame(accepted_vs_rejected_rows)
print(f"[{time.time()-t0:.0f}s] Step 7 (accepted vs rejected) done.")
print(r9_accepted_vs_rejected[["model", "K", "accepted_n", "rejected_n", "accepted_positive_rate",
                                "rejected_positive_rate", "accepted_minus_rejected_return"]].to_string())

# ---------------------------------------------------------------------------

[30s] Step 7 (accepted vs rejected) done.
   model  K  accepted_n  rejected_n  accepted_positive_rate  rejected_positive_rate  accepted_minus_rejected_return
0  NN_v1  2           1        1937                     0.0                0.378936                       -0.000474
1  NN_v1  3           2        2905                     0.0                0.371429                       -0.027805
2  NN_v2  2           0        1938                     NaN                0.394221                             NaN
3  NN_v2  3           0        2907                     NaN                0.381837                             NaN


## 8. Active-week portfolio (Step 16) and calendar-week cash (Step 17) results for TOP-K baseline vs. TOPK_MC_ABSTAIN.

In [9]:
# 8. Active-week portfolio (Step 16) + calendar-cash (Step 17) for
#    TOPK baseline vs TOPK_MC_ABSTAIN.
# ---------------------------------------------------------------------------
def basket_series_from_candidates(cand: pd.DataFrame, full_week_index: pd.DatetimeIndex, only_accepted: bool):
    """Builds a per-week basket series (equal-weight mean of the relevant
    candidates that week) plus the all-sector-mean/SPY context needed for
    the diagnostics -- one row per week in `full_week_index`, using NaN
    (status=NO_TRADE) for weeks with zero relevant candidates."""
    sub = cand[cand["mc_confident"]] if only_accepted else cand
    rows = []
    grouped = {d: g for d, g in sub.groupby("date")}
    for date in full_week_index:
        g = grouped.get(date)
        if g is None or len(g) == 0:
            rows.append({"date": date, "n": 0, "status": "NO_TRADE", "basket_mean_return": np.nan,
                         "basket_median_return": np.nan})
        else:
            rows.append({"date": date, "n": len(g), "status": "ACTIVE",
                         "basket_mean_return": g["next_week_open_to_close_return"].mean(),
                         "basket_median_return": g["next_week_open_to_close_return"].median()})
    out = pd.DataFrame(rows).set_index("date")
    return out


# all-sector mean / SPY per week, from the FULL (11-sector) universe, not just candidates.
all_sector_context = {}
for model_name, full in FULL_BY_MODEL.items():
    ctx_rows = []
    for date, g in full.groupby("date"):
        if len(g) < 11:
            continue
        ctx_rows.append({"date": date, "all_sector_mean_return": g["next_week_open_to_close_return"].mean(),
                          "spy_return": g["next_week_spy_return"].iloc[0]})
    all_sector_context[model_name] = pd.DataFrame(ctx_rows).set_index("date")

active_portfolio_rows = []
calendar_cash_rows = []
basket_series_store = {}  # (model,K,method) -> DataFrame with columns [n,status,basket_mean_return,basket_return_cash,...]

for (model_name, k), cand in candidates.items():
    ctx = all_sector_context[model_name]
    week_index = ctx.index
    for method, only_accepted in [(f"TOP{k}", False), (f"TOP{k}_MC_ABSTAIN", True)]:
        basket = basket_series_from_candidates(cand, week_index, only_accepted)
        basket = basket.join(ctx)
        basket["basket_minus_allsector"] = basket["basket_mean_return"] - basket["all_sector_mean_return"]
        basket["basket_minus_spy"] = basket["basket_mean_return"] - basket["spy_return"]
        basket["win_vs_spy"] = (basket["basket_mean_return"] > basket["spy_return"]).astype(float)
        basket["basket_return_cash"] = np.where(basket["status"] == "ACTIVE", basket["basket_mean_return"], 0.0)
        basket["cash_minus_allsector"] = basket["basket_return_cash"] - basket["all_sector_mean_return"]
        basket["cash_minus_spy"] = basket["basket_return_cash"] - basket["spy_return"]
        basket["cash_win_vs_spy"] = (basket["basket_return_cash"] > basket["spy_return"]).astype(float)
        basket_series_store[(model_name, k, method)] = basket

        active = basket[basket["status"] == "ACTIVE"]
        active_portfolio_rows.append({
            "model": model_name, "K": k, "method": method, "n_active_weeks": len(active),
            "n_total_weeks": len(basket),
            "mean_return": active["basket_mean_return"].mean() if len(active) else np.nan,
            "median_return": active["basket_mean_return"].median() if len(active) else np.nan,
            "minus_allsector_mean": active["basket_minus_allsector"].mean() if len(active) else np.nan,
            "minus_spy_mean": active["basket_minus_spy"].mean() if len(active) else np.nan,
            "win_rate_vs_spy": active["win_vs_spy"].mean() if len(active) else np.nan,
            "pct_active_weeks_positive": (active["basket_mean_return"] > 0).mean() if len(active) else np.nan,
            "pct_active_weeks_ge_1pct": (active["basket_mean_return"] >= 0.01).mean() if len(active) else np.nan,
        })

        compounded = float(np.prod(1 + basket["basket_return_cash"]) - 1)
        calendar_cash_rows.append({
            "model": model_name, "K": k, "method": method, "n_weeks": len(basket),
            "n_no_trade_weeks": int((basket["status"] == "NO_TRADE").sum()),
            "no_trade_pct": float((basket["status"] == "NO_TRADE").mean()),
            "mean_weekly_return": basket["basket_return_cash"].mean(),
            "compounded_return_diagnostic": compounded,
            "mean_minus_spy": basket["cash_minus_spy"].mean(),
            "win_rate_vs_spy": basket["cash_win_vs_spy"].mean(),
        })

r9_active_week_portfolio_metrics = pd.DataFrame(active_portfolio_rows)
r9_calendar_cash_metrics = pd.DataFrame(calendar_cash_rows)
print(f"[{time.time()-t0:.0f}s] Step 8 (active-week + calendar-cash portfolio metrics) done.")

# ---------------------------------------------------------------------------

[30s] Step 8 (active-week + calendar-cash portfolio metrics) done.


## 9. Uncertainty-by-rank diagnostic (full universe, ranks 1-11).

In [10]:
# 9. Uncertainty by deterministic rank (Step 19), full universe (ranks 1-11).
# ---------------------------------------------------------------------------
uncertainty_by_rank_rows = []
for model_name, full in FULL_BY_MODEL.items():
    ranked_rows = []
    for date, g in full.groupby("date"):
        if len(g) < 11:
            continue
        r = g["predicted_proba"].rank(ascending=False, method="first").astype(int)
        tmp = g.copy(); tmp["rank"] = r.to_numpy()
        ranked_rows.append(tmp)
    ranked_full = pd.concat(ranked_rows, ignore_index=True)
    by_rank = ranked_full.groupby("rank").agg(
        mean_mc_std=("mc_std", "mean"), median_mc_std=("mc_std", "median"),
        mean_confidence_mass=("confidence_mass", "mean"), confidence_pass_rate=("mc_confident", "mean"),
        n_obs=("mc_std", "count"),
    ).reset_index()
    by_rank.insert(0, "model", model_name)
    uncertainty_by_rank_rows.append(by_rank)
r9_uncertainty_by_rank = pd.concat(uncertainty_by_rank_rows, ignore_index=True)
print(f"[{time.time()-t0:.0f}s] Step 9 (uncertainty by rank) done.")

# ---------------------------------------------------------------------------

[31s] Step 9 (uncertainty by rank) done.


## 10. Score-margin diagnostic -- does MC uncertainty just reproduce deterministic rank separation?

In [11]:
# 10. Score-margin diagnostic (Step 20).
# ---------------------------------------------------------------------------
margin_rows = []
for model_name, full in FULL_BY_MODEL.items():
    for date, g in full.groupby("date"):
        if len(g) < 11:
            continue
        gs = g.sort_values("predicted_proba", ascending=False).reset_index(drop=True)
        row = {
            "model": model_name, "date": date,
            "score_rank1": gs.loc[0, "predicted_proba"], "score_rank2": gs.loc[1, "predicted_proba"],
            "score_rank3": gs.loc[2, "predicted_proba"], "score_rank4": gs.loc[3, "predicted_proba"],
            "margin_1_2": gs.loc[0, "predicted_proba"] - gs.loc[1, "predicted_proba"],
            "margin_2_3": gs.loc[1, "predicted_proba"] - gs.loc[2, "predicted_proba"],
            "margin_3_4": gs.loc[2, "predicted_proba"] - gs.loc[3, "predicted_proba"],
            "mc_std_rank2": gs.loc[1, "mc_std"], "mc_std_rank3": gs.loc[2, "mc_std"],
            "confidence_mass_rank2": gs.loc[1, "confidence_mass"], "confidence_mass_rank3": gs.loc[2, "confidence_mass"],
        }
        margin_rows.append(row)
r9_score_margin_diagnostic = pd.DataFrame(margin_rows)
corr_2_3 = r9_score_margin_diagnostic.groupby("model").apply(
    lambda d: d["margin_2_3"].corr(d["mc_std_rank2"]), include_groups=False)
corr_3_4 = r9_score_margin_diagnostic.groupby("model").apply(
    lambda d: d["margin_3_4"].corr(d["mc_std_rank3"]), include_groups=False)
print(f"[{time.time()-t0:.0f}s] Step 10 (score-margin diagnostic) done. "
      f"corr(margin_2_3, mc_std_rank2): {corr_2_3.to_dict()}; corr(margin_3_4, mc_std_rank3): {corr_3_4.to_dict()}")

# ---------------------------------------------------------------------------

[31s] Step 10 (score-margin diagnostic) done. corr(margin_2_3, mc_std_rank2): {'NN_v1': 0.3424592351006518, 'NN_v2': 0.23934213936490287}; corr(margin_3_4, mc_std_rank3): {'NN_v1': 0.24060881406967016, 'NN_v2': 0.1168848454938353}


## 11. Uncertainty vs. true/false positives among Top-K candidates.

In [12]:
# 11. Uncertainty vs. errors among Top-K candidates (Step 21).
# ---------------------------------------------------------------------------
error_diag_rows = []
for (model_name, k), cand in candidates.items():
    tp = cand[cand["label_binary"] == 1]
    fp = cand[cand["label_binary"] == 0]
    error_diag_rows.append({
        "model": model_name, "K": k,
        "n_true_positive_candidates": len(tp), "n_false_positive_candidates": len(fp),
        "mean_mc_std_tp": tp["mc_std"].mean() if len(tp) else np.nan,
        "mean_mc_std_fp": fp["mc_std"].mean() if len(fp) else np.nan,
        "mean_confidence_mass_tp": tp["confidence_mass"].mean() if len(tp) else np.nan,
        "mean_confidence_mass_fp": fp["confidence_mass"].mean() if len(fp) else np.nan,
    })
r9_uncertainty_error_diagnostic = pd.DataFrame(error_diag_rows)
print(f"[{time.time()-t0:.0f}s] Step 11 (uncertainty vs. TP/FP) done.")

# ---------------------------------------------------------------------------

[31s] Step 11 (uncertainty vs. TP/FP) done.


## 12. 50-vs-100-pass stability diagnostic (subsampling the first 50 of the same 100 passes).

In [13]:
# 12. 50-vs-100-pass stability diagnostic (Step 22) -- first 50 of the same
#     100 passes, per the predeclared, documented subsampling design.
# ---------------------------------------------------------------------------
stability_rows = []
for model_name, full in FULL_BY_MODEL.items():
    corr_mean = full["mc_std_50"].corr(full["mc_std"])
    corr_cm = full["confidence_mass_50"].corr(full["confidence_mass"])
    mean_abs_diff_std = (full["mc_std_50"] - full["mc_std"]).abs().mean()
    mean_abs_diff_cm = (full["confidence_mass_50"] - full["confidence_mass"]).abs().mean()
    pct_classification_changed = (full["mc_confident_50"] != full["mc_confident"]).mean()
    stability_rows.append({
        "model": model_name, "corr_mc_std_50_vs_100": corr_mean, "corr_confidence_mass_50_vs_100": corr_cm,
        "mean_abs_diff_mc_std": mean_abs_diff_std, "mean_abs_diff_confidence_mass": mean_abs_diff_cm,
        "pct_confidence_classification_changed": pct_classification_changed,
    })
r9_mc_pass_stability = pd.DataFrame(stability_rows)
print(f"[{time.time()-t0:.0f}s] Step 12 (50-vs-100-pass stability) done.")
print(r9_mc_pass_stability.to_string())

# ---------------------------------------------------------------------------

[31s] Step 12 (50-vs-100-pass stability) done.
   model  corr_mc_std_50_vs_100  corr_confidence_mass_50_vs_100  mean_abs_diff_mc_std  mean_abs_diff_confidence_mass  pct_confidence_classification_changed
0  NN_v1               0.990387                        0.593261              0.002087                       0.031686                               0.002531
1  NN_v2               0.984225                        0.520493              0.001167                       0.030046                               0.004874


## 13. Block bootstrap: standalone CIs, paired MC-abstention-vs-baseline, and accepted-vs-rejected weekly-paired comparisons.

In [14]:
# 13. Block bootstrap (Step 23): standalone + paired vs TOPK baseline +
#     accepted-minus-rejected (weekly-paired, only weeks with both groups).
# ---------------------------------------------------------------------------
block_bootstrap_ci = r7.block_bootstrap_ci

bootstrap_rows = []
for (model_name, k) in candidates:
    for method in [f"TOP{k}", f"TOP{k}_MC_ABSTAIN"]:
        basket = basket_series_store[(model_name, k, method)]
        active = basket[basket["status"] == "ACTIVE"]
        if len(active) >= BLOCK_SIZE:
            b = block_bootstrap_ci(active["basket_minus_allsector"])
            bootstrap_rows.append({"model": model_name, "K": k, "method": method, "variant": "active",
                                    "quantity": "basket_minus_allsector", **b})
            b = block_bootstrap_ci(active["basket_minus_spy"])
            bootstrap_rows.append({"model": model_name, "K": k, "method": method, "variant": "active",
                                    "quantity": "basket_minus_spy", **b})
            b = block_bootstrap_ci(active["win_vs_spy"] - 0.5)
            bootstrap_rows.append({"model": model_name, "K": k, "method": method, "variant": "active",
                                    "quantity": "win_rate_edge_vs_spy", **b})
        if len(basket) >= BLOCK_SIZE:
            b = block_bootstrap_ci(basket["cash_minus_spy"])
            bootstrap_rows.append({"model": model_name, "K": k, "method": method, "variant": "calendar_cash",
                                    "quantity": "cash_minus_spy", **b})
            b = block_bootstrap_ci(basket["cash_win_vs_spy"] - 0.5)
            bootstrap_rows.append({"model": model_name, "K": k, "method": method, "variant": "calendar_cash",
                                    "quantity": "win_rate_edge_vs_spy", **b})
r9_bootstrap_ci = pd.DataFrame(bootstrap_rows)
print(f"[{time.time()-t0:.0f}s] Step 13a (standalone bootstrap) done.")


def paired_diff_bootstrap(series_a: pd.Series, series_b: pd.Series):
    common_idx = series_a.dropna().index.intersection(series_b.dropna().index)
    if len(common_idx) < BLOCK_SIZE:
        return None
    diff = (series_a.loc[common_idx] - series_b.loc[common_idx]).sort_index()
    return block_bootstrap_ci(diff)


paired_vs_topk_rows = []
for (model_name, k) in candidates:
    mc_basket = basket_series_store[(model_name, k, f"TOP{k}_MC_ABSTAIN")]
    base_basket = basket_series_store[(model_name, k, f"TOP{k}")]
    for qname, col, variant in [
        ("basket_minus_allsector_diff", "basket_minus_allsector", "active"),
        ("basket_minus_spy_diff", "basket_minus_spy", "active"),
        ("win_rate_diff", "win_vs_spy", "active"),
    ]:
        mc_active = mc_basket[mc_basket["status"] == "ACTIVE"]
        base_active = base_basket[base_basket["status"] == "ACTIVE"]
        b = paired_diff_bootstrap(mc_active[col], base_active[col])
        if b is not None:
            paired_vs_topk_rows.append({"model": model_name, "K": k, "vs_baseline": f"TOP{k}",
                                         "variant": variant, "quantity": qname, **b})
    for qname, col in [("cash_minus_spy_diff", "cash_minus_spy"), ("cash_win_rate_diff", "cash_win_vs_spy")]:
        b = paired_diff_bootstrap(mc_basket[col], base_basket[col])
        if b is not None:
            paired_vs_topk_rows.append({"model": model_name, "K": k, "vs_baseline": f"TOP{k}",
                                         "variant": "calendar_cash", "quantity": qname, **b})

    cand = candidates[(model_name, k)]
    accepted_by_week = cand[cand["mc_confident"]].groupby("date")["next_week_open_to_close_return"].mean()
    rejected_by_week = cand[~cand["mc_confident"]].groupby("date")["next_week_open_to_close_return"].mean()
    b = paired_diff_bootstrap(accepted_by_week, rejected_by_week)
    if b is not None:
        paired_vs_topk_rows.append({"model": model_name, "K": k, "vs_baseline": "rejected_candidates",
                                     "variant": "weekly_paired", "quantity": "accepted_minus_rejected_return", **b})

    cand_excess = cand.assign(excess=cand["next_week_open_to_close_return"] - cand["next_week_spy_return"])
    accepted_excess_wk = cand_excess[cand_excess["mc_confident"]].groupby("date")["excess"].mean()
    rejected_excess_wk = cand_excess[~cand_excess["mc_confident"]].groupby("date")["excess"].mean()
    b = paired_diff_bootstrap(accepted_excess_wk, rejected_excess_wk)
    if b is not None:
        paired_vs_topk_rows.append({"model": model_name, "K": k, "vs_baseline": "rejected_candidates",
                                     "variant": "weekly_paired", "quantity": "accepted_minus_rejected_excess_spy", **b})
    print(f"[{time.time()-t0:.0f}s] {model_name} K={k}: paired bootstrap done "
          f"(n_common_weeks_accepted_vs_rejected={len(accepted_by_week.index.intersection(rejected_by_week.index))})")

r9_vs_topk_paired_comparison = pd.DataFrame(paired_vs_topk_rows)
print(f"[{time.time()-t0:.0f}s] Step 13b (paired vs TOPK + accepted-vs-rejected bootstrap) done.")

# ---------------------------------------------------------------------------

[36s] Step 13a (standalone bootstrap) done.


[37s] NN_v1 K=2: paired bootstrap done (n_common_weeks_accepted_vs_rejected=1)


[37s] NN_v1 K=3: paired bootstrap done (n_common_weeks_accepted_vs_rejected=2)


[37s] NN_v2 K=2: paired bootstrap done (n_common_weeks_accepted_vs_rejected=0)


[38s] NN_v2 K=3: paired bootstrap done (n_common_weeks_accepted_vs_rejected=0)
[38s] Step 13b (paired vs TOPK + accepted-vs-rejected bootstrap) done.


## 14. Sample audit.

In [15]:
# 14. Sample audit.
# ---------------------------------------------------------------------------
sample_audit_rows = []
for model_name, full in FULL_BY_MODEL.items():
    sample_audit_rows.append({
        "model": model_name, "n_oos_rows": len(full), "n_weeks": full["date"].nunique(),
        "first_date": str(full["date"].min().date()), "last_date": str(full["date"].max().date()),
        "mc_passes_primary": r9.MC_PASSES_PRIMARY, "mc_passes_stability": r9.MC_PASSES_STABILITY,
        "confidence_mass_threshold": r9.CONFIDENCE_MASS_THRESHOLD, "confidence_std_multiplier": r9.CONFIDENCE_STD_MULTIPLIER,
        "underlying_auc": float(roc_auc_score(full.label_binary, full.predicted_proba)),
        "overall_confidence_pass_rate": float(full["mc_confident"].mean()),
    })
r9_sample_audit = pd.DataFrame(sample_audit_rows)

# ---------------------------------------------------------------------------

## 15. Save all required R9 artifacts.

In [16]:
# 15. Save all artifacts.
# ---------------------------------------------------------------------------
r9_mc_prediction_summary.to_csv(OUTPUTS_ROOT / "r9_mc_prediction_summary.csv", index=False)
r9_confidence_decisions.to_csv(OUTPUTS_ROOT / "r9_confidence_decisions.csv", index=False)
for k in K_VALUES:
    cols_order = ["model", "K"]
    combined = pd.concat([
        r9_coverage[r9_coverage.K == k],
    ], ignore_index=True)
    fname = f"r9_top{k}_metrics.csv"
    # Combine coverage + accepted/rejected + active/cash portfolio for this K into one wide report table.
    merged = r9_coverage[r9_coverage.K == k].merge(
        r9_accepted_vs_rejected[r9_accepted_vs_rejected.K == k], on=["model", "K"], how="left")
    active_k = r9_active_week_portfolio_metrics[r9_active_week_portfolio_metrics.K == k]
    cash_k = r9_calendar_cash_metrics[r9_calendar_cash_metrics.K == k]
    merged.to_csv(OUTPUTS_ROOT / f"r9_coverage_and_classification_k{k}.csv", index=False)
    if k == 2:
        r9_top2_metrics = merged.merge(active_k, on=["model", "K"], how="left", suffixes=("", "_active"))
    else:
        r9_top3_metrics = merged.merge(active_k, on=["model", "K"], how="left", suffixes=("", "_active"))

r9_top2_metrics.to_csv(OUTPUTS_ROOT / "r9_top2_metrics.csv", index=False)
r9_top3_metrics.to_csv(OUTPUTS_ROOT / "r9_top3_metrics.csv", index=False)
r9_accepted_vs_rejected.to_csv(OUTPUTS_ROOT / "r9_accepted_vs_rejected.csv", index=False)
r9_active_week_portfolio_metrics.to_csv(OUTPUTS_ROOT / "r9_active_week_portfolio_metrics.csv", index=False)
r9_calendar_cash_metrics.to_csv(OUTPUTS_ROOT / "r9_calendar_cash_metrics.csv", index=False)
r9_uncertainty_by_rank.to_csv(OUTPUTS_ROOT / "r9_uncertainty_by_rank.csv", index=False)
r9_uncertainty_error_diagnostic.to_csv(OUTPUTS_ROOT / "r9_uncertainty_error_diagnostic.csv", index=False)
r9_score_margin_diagnostic.to_csv(OUTPUTS_ROOT / "r9_score_margin_diagnostic.csv", index=False)
r9_mc_pass_stability.to_csv(OUTPUTS_ROOT / "r9_mc_pass_stability.csv", index=False)
r9_bootstrap_ci.to_csv(OUTPUTS_ROOT / "r9_bootstrap_ci.csv", index=False)
r9_vs_topk_paired_comparison.to_csv(OUTPUTS_ROOT / "r9_vs_topk_paired_comparison.csv", index=False)
r9_sample_audit.to_csv(OUTPUTS_ROOT / "r9_sample_audit.csv", index=False)
r9_pass_rate_by_etf.to_csv(OUTPUTS_ROOT / "r9_pass_rate_by_etf.csv", index=False)
r9_pass_rate_by_year.to_csv(OUTPUTS_ROOT / "r9_pass_rate_by_year.csv", index=False)
r9_confidence_mass_distribution.to_csv(OUTPUTS_ROOT / "r9_confidence_mass_distribution.csv", index=False)

overall_pass_rate = {m: float(FULL_BY_MODEL[m]["mc_confident"].mean()) for m in ["NN_v1", "NN_v2"]}
degenerate = all(v < 0.01 for v in overall_pass_rate.values()) or all(v > 0.99 for v in overall_pass_rate.values())

summary = {
    "mc_passes_primary": r9.MC_PASSES_PRIMARY, "mc_passes_stability": r9.MC_PASSES_STABILITY,
    "confidence_mass_threshold": r9.CONFIDENCE_MASS_THRESHOLD, "confidence_std_multiplier": r9.CONFIDENCE_STD_MULTIPLIER,
    "underlying_auc": {"NN_v1": float(auc_v1), "NN_v2": float(auc_v2)},
    "overall_confidence_pass_rate": overall_pass_rate,
    "coverage_summary": r9_coverage.set_index(["model", "K"])[["acceptance_rate", "mean_accepted_per_week",
                                                                  "active_week_pct"]].reset_index().to_dict("records"),
    "rule_appears_degenerate": degenerate,
    "runtime_seconds": time.time() - t0,
}
with open(OUTPUTS_ROOT / "r9_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)

print(f"[{time.time()-t0:.0f}s] ALL DONE.")
print(json.dumps(summary, indent=2, default=str))

[38s] ALL DONE.
{
  "mc_passes_primary": 100,
  "mc_passes_stability": 50,
  "confidence_mass_threshold": 0.8,
  "confidence_std_multiplier": 1.0,
  "underlying_auc": {
    "NN_v1": 0.5680100994943317,
    "NN_v2": 0.5735730758971164
  },
  "overall_confidence_pass_rate": {
    "NN_v1": 0.0008435654700534258,
    "NN_v2": 0.0
  },
  "coverage_summary": [
    {
      "model": "NN_v1",
      "K": 2,
      "acceptance_rate": 0.0005159958720330237,
      "mean_accepted_per_week": 0.0010319917440660474,
      "active_week_pct": 0.0010319917440660474
    },
    {
      "model": "NN_v1",
      "K": 3,
      "acceptance_rate": 0.0006879944960440316,
      "mean_accepted_per_week": 0.0020639834881320948,
      "active_week_pct": 0.0020639834881320948
    },
    {
      "model": "NN_v2",
      "K": 2,
      "acceptance_rate": 0.0,
      "mean_accepted_per_week": 0.0,
      "active_week_pct": 0.0
    },
    {
      "model": "NN_v2",
      "K": 3,
      "acceptance_rate": 0.0,
      "mean_accepted